## 1 · Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'font.size': 10, 'axes.titlesize': 11})
rng = np.random.default_rng(0)

DATA_DIR = Path('../../data')
AUTH_FILE = DATA_DIR / 'raman_valid_samples_combined_aligned_402_3499.xlsx'
ADU_FILE  = DATA_DIR / 'raman_functional_intensity_matrix_bspline_n4_aligned_402_3499.xlsx'
SHEET = 'Aligned Spectra'

# DWT convention — identical to raman_decomposition.ipynb
WAVELET  = 'db4'
N_LEVELS = 6
DWT_BAND = (5, 6)          # detail levels used for the "D5:D6" consensus
print('Config ready.')

In [ ]:
auth_df = pd.read_excel(AUTH_FILE, sheet_name=SHEET)
adu_df  = pd.read_excel(ADU_FILE,  sheet_name=SHEET)

# First column is the wavenumber axis
x = auth_df.iloc[:, 0].to_numpy(dtype=float)
assert np.array_equal(x, adu_df.iloc[:, 0].to_numpy(dtype=float)), 'grids differ!'

# Columns = samples; rows = wavenumbers. Store spectra as rows -> shape (n_samples, n_wn)
auth_names = list(auth_df.columns[1:])
adu_names  = list(adu_df.columns[1:])
X_auth = auth_df.iloc[:, 1:].to_numpy(dtype=float).T   # (44, 3098)
X_adu  = adu_df.iloc[:, 1:].to_numpy(dtype=float).T     # (39, 3098)
dx = float(np.median(np.diff(x)))

print(f'Wavenumber axis : {x[0]:.0f}–{x[-1]:.0f} cm⁻¹, {len(x)} points, spacing ≈ {dx:.2f} cm⁻¹')
print(f'Authentic       : {X_auth.shape[0]} spectra')
print(f'Adulterated     : {X_adu.shape[0]} spectra')

## 2 · Consensus reference builders

In [ ]:
def arithmetic_mean(X):
    """Per-wavenumber arithmetic mean. X: (n_samples, n_wn) -> (n_wn,)"""
    return X.mean(axis=0)


def geometric_median(X, eps=1e-8, max_iter=500, tol=1e-9):
    """Weiszfeld geometric median of rows of X (robust multivariate centre)."""
    y = X.mean(axis=0)
    for _ in range(max_iter):
        d = np.linalg.norm(X - y, axis=1)
        d = np.where(d < eps, eps, d)
        w = 1.0 / d
        y_new = (w[:, None] * X).sum(axis=0) / w.sum()
        if np.linalg.norm(y_new - y) < tol:
            y = y_new
            break
        y = y_new
    return y


def dwt_band_reconstruct(y, wavelet=WAVELET, levels=N_LEVELS, band=DWT_BAND):
    """Reconstruct signal keeping only the given DWT detail levels (e.g. D5+D6).

    Uses the same convention as raman_decomposition.ipynb: coeffs = [cA_n, cD_n, ..., cD_1].
    Detail level k lives at index (levels + 1 - k).
    """
    coeffs = pywt.wavedec(y, wavelet, level=levels)
    kept = [np.zeros_like(c) for c in coeffs]
    for k in band:                       # k = detail level number (1=finest .. levels=coarsest)
        idx = levels + 1 - k
        kept[idx] = coeffs[idx]
    rec = pywt.waverec(kept, wavelet)
    return rec[:len(y)]


## 3 · Build the three consensus references from the authentic set

In [ ]:
c_mean  = arithmetic_mean(X_auth)
c_gmed  = geometric_median(X_auth)
# D5:D6 consensus = wavelet band of the arithmetic-mean consensus
c_dwt56 = dwt_band_reconstruct(c_mean)

references = {
    'Arithmetic Mean':  c_mean,
    'Geometric Median': c_gmed,
    'DWT D5:D6':        c_dwt56,
}
print('Consensus references built:', list(references))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(x, c_mean, label='Arithmetic Mean', lw=1)
ax[0].plot(x, c_gmed, label='Geometric Median', lw=1, ls='--')
ax[0].set_title('Full-spectrum consensus references')
ax[0].set_xlabel('Raman shift (cm⁻¹)'); ax[0].set_ylabel('Intensity'); ax[0].legend()
ax[1].plot(x, c_dwt56, color='crimson', lw=0.9)
ax[1].axhline(0, color='grey', lw=0.4, ls='--')
ax[1].set_title(f'DWT D5:D6 band of consensus (~{dx*2**5:.0f} & {dx*2**6:.0f} cm⁻¹ features)')
ax[1].set_xlabel('Raman shift (cm⁻¹)'); ax[1].set_ylabel('Amplitude')
plt.tight_layout(); plt.show()

## 4 · Distance functions

For a spectrum *s* the raw distances are

- `d_mean(s)  = ‖s − c_mean‖₂`
- `d_gmed(s)  = ‖s − c_gmed‖₂`
- `d_dwt56(s) = ‖DWT56(s) − c_dwt56‖₂`  — the sample is first projected onto its own D5:D6 band,
  then compared to the consensus D5:D6 band.


In [ ]:
def raw_distances(s):
    """Return dict of raw L2 distances of spectrum s to each reference."""
    return {
        'Arithmetic Mean':  np.linalg.norm(s - c_mean),
        'Geometric Median': np.linalg.norm(s - c_gmed),
        'DWT D5:D6':        np.linalg.norm(dwt_band_reconstruct(s) - c_dwt56),
    }

D_auth = pd.DataFrame([raw_distances(s) for s in X_auth], index=auth_names)
D_adu  = pd.DataFrame([raw_distances(s) for s in X_adu],  index=adu_names)

print('Authentic raw distances (summary):')
print(D_auth.describe().loc[['mean', 'std', 'min', 'max']].round(5))

## 5 · Leave-one-out sensitivity weights

Drop each authentic sample in turn, recompute every consensus, and record the **relative shift**
`‖c_full − c₋ᵢ‖ / ‖c_full‖`. Averaging over all left-out samples gives each reference's LOO
*sensitivity* — how much it responds to the composition of the authentic set. Weights are these
sensitivities normalised to sum to 1 (**more sensitive ⇒ higher weight**, the chosen scheme).

Relative (rather than absolute) shifts are used so the three references — which live on very
different amplitude scales — are compared fairly.

In [ ]:
n = X_auth.shape[0]
shifts = {k: np.empty(n) for k in references}
norm_full = {k: np.linalg.norm(v) for k, v in references.items()}

for i in range(n):
    Xmi = np.delete(X_auth, i, axis=0)          # leave sample i out
    m_i   = arithmetic_mean(Xmi)
    g_i   = geometric_median(Xmi)
    d56_i = dwt_band_reconstruct(m_i)
    shifts['Arithmetic Mean'][i]  = np.linalg.norm(c_mean  - m_i)   / norm_full['Arithmetic Mean']
    shifts['Geometric Median'][i] = np.linalg.norm(c_gmed  - g_i)   / norm_full['Geometric Median']
    shifts['DWT D5:D6'][i]        = np.linalg.norm(c_dwt56 - d56_i) / norm_full['DWT D5:D6']

sensitivity = {k: shifts[k].mean() for k in references}
total_sens  = sum(sensitivity.values())
weights = {k: sensitivity[k] / total_sens for k in references}

loo_tbl = pd.DataFrame({
    'Mean rel. LOO shift': sensitivity,
    'Weight':              weights,
}).round(6)
print(loo_tbl)
print(f"\nWeights sum to {sum(weights.values()):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
labels = list(references)
ax[0].bar(labels, [sensitivity[k] for k in labels], color=['#4C72B0', '#55A868', '#C44E52'])
ax[0].set_title('Mean relative LOO shift (sensitivity)'); ax[0].set_ylabel('relative shift')
ax[0].tick_params(axis='x', rotation=15)
ax[1].bar(labels, [weights[k] for k in labels], color=['#4C72B0', '#55A868', '#C44E52'])
ax[1].set_title('Normalised weights'); ax[1].set_ylabel('weight')
ax[1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

## 6 · Authenticity score

Standardise each distance against the **authentic baseline** (mean μᵣ, std σᵣ over the 44 authentic
samples), then combine with the LOO weights:

$$\text{score}(s) = \sum_r w_r \, z_r(s), \qquad z_r(s) = \frac{d_r(s) - \mu_r}{\sigma_r}$$

By construction the authentic set has mean score ≈ 0. A larger score means the spectrum is
systematically further from the authentic consensus than a typical authentic sample.

In [ ]:
mu    = D_auth.mean()
sigma = D_auth.std(ddof=1)

def zscore(df):
    return (df - mu) / sigma

Z_auth = zscore(D_auth)
Z_adu  = zscore(D_adu)
w_vec  = pd.Series(weights)

score_auth = (Z_auth * w_vec).sum(axis=1)
score_adu  = (Z_adu  * w_vec).sum(axis=1)

print(f'Authentic score  : mean {score_auth.mean():+.3f}, std {score_auth.std():.3f}, '
      f'range [{score_auth.min():+.3f}, {score_auth.max():+.3f}]')
print(f'Adulterated score: mean {score_adu.mean():+.3f}, std {score_adu.std():.3f}, '
      f'range [{score_adu.min():+.3f}, {score_adu.max():+.3f}]')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
bins = np.linspace(min(score_auth.min(), score_adu.min()),
                   max(score_auth.max(), score_adu.max()), 30)
ax.hist(score_auth, bins=bins, alpha=0.6, label=f'Authentic (n={len(score_auth)})', color='#55A868')
ax.hist(score_adu,  bins=bins, alpha=0.6, label=f'Adulterated (n={len(score_adu)})', color='#C44E52')
ax.set_xlabel('Authenticity score  (Σ wᵣ · zᵣ)'); ax.set_ylabel('count')
ax.set_title('Authenticity-score distributions'); ax.legend()
plt.tight_layout(); plt.show()

## 7 · Threshold selection for QC

Label authentic = 0, adulterated = 1. We use **Youden's J** — the ROC point maximising
(sensitivity + specificity − 1) — as the single QC threshold, giving the best overall separation
between authentic and adulterated milk.

In [ ]:
y_true  = np.r_[np.zeros(len(score_auth)), np.ones(len(score_adu))]
y_score = np.r_[score_auth.values, score_adu.values]

auc = roc_auc_score(y_true, y_score)
fpr, tpr, thr = roc_curve(y_true, y_score)
youden = thr[np.argmax(tpr - fpr)]

print(f'ROC AUC              : {auc:.4f}')
print(f"Youden's J threshold : {youden:+.4f}")

In [ ]:
def qc_report(t, name):
    pred = (y_score > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    sens = tp / (tp + fn)                 # adulterated correctly flagged
    spec = tn / (tn + fp)                 # authentic correctly passed
    acc  = (tp + tn) / len(y_true)
    print(f'--- {name}  (threshold = {t:+.4f}) ---')
    print(f'  Sensitivity (adulterated caught): {sens:6.1%}   [{tp}/{tp+fn}]')
    print(f'  Specificity (authentic passed)  : {spec:6.1%}   [{tn}/{tn+fp}]')
    print(f'  Accuracy                        : {acc:6.1%}')
    print(f'  Confusion [tn fp / fn tp]       : [{tn} {fp} / {fn} {tp}]\n')
    return dict(threshold=t, sensitivity=sens, specificity=spec, accuracy=acc,
               tn=tn, fp=fp, fn=fn, tp=tp)

r_youden = qc_report(youden, "Youden's J")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(fpr, tpr, color='#4C72B0', lw=2, label=f'ROC (AUC = {auc:.3f})')
ax[0].plot([0, 1], [0, 1], ls='--', color='grey', lw=1)
jx = np.argmax(tpr - fpr)
ax[0].scatter(fpr[jx], tpr[jx], color='crimson', zorder=5, label="Youden's J")
ax[0].set_xlabel('False-positive rate'); ax[0].set_ylabel('True-positive rate')
ax[0].set_title('ROC — authentic vs adulterated'); ax[0].legend(loc='lower right')

order = np.argsort(y_score)
colors = np.where(y_true[order] == 1, '#C44E52', '#55A868')
ax[1].scatter(range(len(y_score)), y_score[order], c=colors, s=22)
ax[1].axhline(youden, color='k', ls='-', lw=1, label=f"Youden {youden:+.2f}")
ax[1].set_xlabel('samples (sorted by score)'); ax[1].set_ylabel('authenticity score')
ax[1].set_title('Score ranking (green=authentic, red=adulterated)'); ax[1].legend()
plt.tight_layout(); plt.show()

## 8 · Export per-sample results

In [ ]:
def assemble(scores, Z, D, label):
    out = pd.DataFrame(index=scores.index)
    out['class'] = label
    for k in references:
        out[f'd_{k}'] = D[k]
    for k in references:
        out[f'z_{k}'] = Z[k]
    out['authenticity_score'] = scores
    out['flag_youden'] = (scores > youden).astype(int)
    return out

results = pd.concat([
    assemble(score_auth, Z_auth, D_auth, 'authentic'),
    assemble(score_adu,  Z_adu,  D_adu,  'adulterated'),
])
results.index.name = 'sample'

weights_out = loo_tbl.copy()
summary_out = pd.DataFrame({
    'metric': ['ROC_AUC', 'youden_threshold',
               'youden_sensitivity', 'youden_specificity', 'youden_accuracy'],
    'value':  [auc, youden,
               r_youden['sensitivity'], r_youden['specificity'], r_youden['accuracy']],
})

OUT = 'raman_loo_authenticity_scores.xlsx'
with pd.ExcelWriter(OUT) as xw:
    results.to_excel(xw, sheet_name='Per-sample scores')
    weights_out.to_excel(xw, sheet_name='LOO weights')
    summary_out.to_excel(xw, sheet_name='QC summary', index=False)
print(f'Saved -> {OUT}')
results.head()

<!--EVAL-->
## 10 · External evaluation on the held-out evaluation dataset

We now apply the **already-fitted** model — the consensus references, the authentic baseline
`(mu, sigma)`, the LOO `weights`, and the two thresholds — to
`Eval_Preprocessed_aligned_402_3499.xlsx` **without any refitting**.

**Ground-truth labelling rule (per request):** a sample is labelled *adulterated* if its name
mentions an adulterant (starch, urea, dilution, vegetable oil); otherwise it is *authentic*.

In [ ]:
EVAL_FILE = DATA_DIR / 'Eval_Preprocessed_aligned_402_3499_recalibrated.xlsx'
eval_df = pd.read_excel(EVAL_FILE, sheet_name=SHEET)
assert np.array_equal(x, eval_df.iloc[:, 0].to_numpy(dtype=float)), 'eval grid differs!'

eval_names = list(eval_df.columns[1:])
X_eval = eval_df.iloc[:, 1:].to_numpy(dtype=float).T     # (n_eval, 3098)

ADULTERANT_KW = ['starch', 'urea', 'dilution', 'oil', 'veg']
def is_adulterated(name):
    low = name.lower()
    return int(any(k in low for k in ADULTERANT_KW))

eval_true = np.array([is_adulterated(n) for n in eval_names])
for n, t in zip(eval_names, eval_true):
    print(f'  {"ADULTERATED" if t else "authentic  "}  |  {n}')
print(f'\n{eval_true.sum()} adulterated, {(1-eval_true).sum()} authentic  (n={len(eval_names)})')

In [ ]:
# Score with the frozen model: raw distances -> standardise on authentic baseline -> weight
D_eval = pd.DataFrame([raw_distances(s) for s in X_eval], index=eval_names)
Z_eval = zscore(D_eval)                      # uses authentic-set mu, sigma
score_eval = (Z_eval * w_vec).sum(axis=1)    # uses LOO weights

eval_out = pd.DataFrame({
    'true_label': np.where(eval_true == 1, 'adulterated', 'authentic'),
    'authenticity_score': score_eval,
    'pred_youden': np.where(score_eval.values > youden, 'adulterated', 'authentic'),
}, index=eval_names)
eval_out.index.name = 'sample'
eval_out.round(3)

In [ ]:
def eval_report(t, name):
    pred = (score_eval.values > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(eval_true, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else float('nan')
    spec = tn / (tn + fp) if (tn + fp) else float('nan')
    acc  = (tp + tn) / len(eval_true)
    print(f'--- {name}  (threshold = {t:+.4f}) ---')
    print(f'  Sensitivity (adulterated caught): {sens:6.1%}   [{tp}/{tp+fn}]')
    print(f'  Specificity (authentic passed)  : {spec:6.1%}   [{tn}/{tn+fp}]')
    print(f'  Accuracy                        : {acc:6.1%}   [{tp+tn}/{len(eval_true)}]')
    print(f'  Confusion [tn fp / fn tp]       : [{tn} {fp} / {fn} {tp}]\n')

# AUC only defined if both classes present
if len(np.unique(eval_true)) == 2:
    print(f'Eval ROC AUC (ranking quality): {roc_auc_score(eval_true, score_eval.values):.4f}\n')
eval_report(youden, "Youden's J")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
order = np.argsort(score_eval.values)
names_o = np.array(eval_names)[order]
scores_o = score_eval.values[order]
colors = np.where(eval_true[order] == 1, '#C44E52', '#55A868')
ax.barh(range(len(scores_o)), scores_o, color=colors)
ax.axvline(youden, color='k', ls='-', lw=1.2, label=f"Youden {youden:+.2f}")
ax.set_yticks(range(len(names_o))); ax.set_yticklabels(names_o, fontsize=8)
ax.set_xlabel('authenticity score'); ax.set_title(
    "Evaluation samples (red = adulterated, green = authentic) vs Youden's J threshold")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
EVAL_OUT = 'raman_loo_eval_results.xlsx'
with pd.ExcelWriter(EVAL_OUT) as xw:
    full = eval_out.copy()
    for k in references:
        full[f'd_{k}'] = D_eval[k]
        full[f'z_{k}'] = Z_eval[k]
    full.to_excel(xw, sheet_name='Eval scores')
print(f'Saved -> {EVAL_OUT}')

<!--OVERFIT-->
## 11 · Is the LOO weighting overfitting to the training set?

A natural worry after the eval result is that the **leave-one-out weights** are tuned to the 44
authentic training spectra and therefore fail to transfer. This section tests that directly.

The key observation is that the LOO step produces only **three weights that sum to one** — i.e. two
free parameters — and everything else that shapes the score (the consensus references and the
`mu/sigma` standardisation) is *not* set by the LOO. If the LOO weighting were the thing causing the
eval failure, then **replacing it with flat equal weights (1/3 each) should visibly change the
results**. We check both the internal split and the external evaluation set under the two weightings.

In [ ]:
w_equal = pd.Series({k: 1.0 / len(references) for k in references})

# Internal split (train authentic vs adulterated) under both weightings
score_auth_eq = (Z_auth * w_equal).sum(axis=1)
score_adu_eq  = (Z_adu  * w_equal).sum(axis=1)
y_score_eq    = np.r_[score_auth_eq.values, score_adu_eq.values]

# External evaluation set under both weightings
score_eval_eq = (Z_eval * w_equal).sum(axis=1)

auc_int_loo = roc_auc_score(y_true, y_score)
auc_int_eq  = roc_auc_score(y_true, y_score_eq)
auc_ev_loo  = roc_auc_score(eval_true, score_eval.values)
auc_ev_eq   = roc_auc_score(eval_true, score_eval_eq.values)

cmp = pd.DataFrame({
    'LOO weights': [weights['Arithmetic Mean'], weights['Geometric Median'],
                    weights['DWT D5:D6'], auc_int_loo, auc_ev_loo],
    'Equal weights': [1/3, 1/3, 1/3, auc_int_eq, auc_ev_eq],
}, index=['weight: Arithmetic Mean', 'weight: Geometric Median', 'weight: DWT D5:D6',
          'Internal ROC AUC', 'Eval ROC AUC'])
print(cmp.round(4))

max_dev = float((w_vec - 1/len(references)).abs().max())
corr_int  = float(np.corrcoef(y_score, y_score_eq)[0, 1])
corr_eval = float(np.corrcoef(score_eval.values, score_eval_eq.values)[0, 1])
print(f'\nMax deviation of any LOO weight from equal (1/3): {max_dev:.4f}')
print(f'Correlation of per-sample scores (LOO vs equal) — internal: {corr_int:.4f}, eval: {corr_eval:.4f}')
print(f'Internal AUC change LOO->equal: {auc_int_eq - auc_int_loo:+.4f}')
print(f'Eval     AUC change LOO->equal: {auc_ev_eq  - auc_ev_loo:+.4f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# (a) weights are close to the equal-weight line
labels = list(references)
xp = np.arange(len(labels))
ax[0].bar(xp - 0.18, [weights[k] for k in labels], width=0.36, label='LOO weights', color='#4C72B0')
ax[0].bar(xp + 0.18, [1/3]*len(labels),           width=0.36, label='Equal weights', color='#C0C0C0')
ax[0].axhline(1/3, color='k', ls='--', lw=0.8)
ax[0].set_xticks(xp); ax[0].set_xticklabels(labels, rotation=15, fontsize=8)
ax[0].set_ylabel('weight'); ax[0].set_title('LOO weights vs equal weights'); ax[0].legend()

# (b) per-sample scores are near-identical under the two weightings
alln_loo = np.r_[y_score, score_eval.values]
alln_eq  = np.r_[y_score_eq, score_eval_eq.values]
ax[1].scatter(alln_eq, alln_loo, s=22, color='#55A868', alpha=0.8)
lo, hi = alln_eq.min(), alln_eq.max()
ax[1].plot([lo, hi], [lo, hi], ls='--', color='k', lw=1, label='y = x')
ax[1].set_xlabel('score (equal weights)'); ax[1].set_ylabel('score (LOO weights)')
ax[1].set_title(f'Per-sample scores agree (r={np.corrcoef(alln_loo, alln_eq)[0,1]:.3f})')
ax[1].legend()
plt.tight_layout(); plt.show()

## 12 · Bootstrap 95% CI for reported AUCs

In [ ]:
# Bootstrap 95% CI for the reported ROC AUCs (stratified resampling of the fixed scores and labels)
def auc_ci(y_true, y_score, n_boot=5000, alpha=0.05, seed=0):
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    rb = np.random.default_rng(seed)
    i0, i1 = np.flatnonzero(y_true == 0), np.flatnonzero(y_true == 1)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        ii = np.concatenate([rb.choice(i0, i0.size, replace=True),
                             rb.choice(i1, i1.size, replace=True)])
        boot[b] = roc_auc_score(y_true[ii], y_score[ii])
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return roc_auc_score(y_true, y_score), lo, hi

auc_ci_df = pd.DataFrame(
    [auc_ci(y_true, y_score), auc_ci(eval_true, score_eval.values)],
    columns=['AUC', 'CI 2.5%', 'CI 97.5%'],
    index=['Internal ROC AUC', 'Eval ROC AUC'],
)
print(auc_ci_df.round(4))